In [3]:
import sys

import sys
from cells_simulation import *

import sys
sys.path.insert(0, "../../")

from RealTrace_paper_analysis.plotting_src.read_ggp_run import * # you might need to change these two line, such that you can use read ggp_run

import numpy as np
import subprocess

In [ ]:

# ========== Simulation parameters ========== #
save_dataset = True    # saves data set as csv
run_ggp = True         # runs the ggp code (located in "../bin/" relative to this nb)

dt = 0.01
dt_measument = 3 # in minutes

n_cells = 1000 # number of cells that will be simulated
discard_cells = 20
lanes = 1

log_length0 = np.log(2)
gfp0 = 24975.8


div_mode = "adder"
division_log_length = None   # for sizer: division, when log_length hits division_log_length
division_time = None          # for timer: division, when cell cycle time hits division_time
division_addition = log_length0       # for adder: divsion, when division_addition in log_length was added in cell cycle


data_set_name = "glucose_hi1"
# taken from glucose_hi1_f01234578910_b_prediction in data_dany_old_data_output_20221215
parameter_sets  ={"glucose_hi1": {  "mean_lambda": 0.0114137, 
                                    "gamma_lambda": 0.166994,
                                    "var_lambda": 4.9481e-06,
                                    "mean_q": 118.524,
                                    "gamma_q": 0.0595599,
                                    "var_q": 430.055,
                                    "beta": 0, #0.0032,
                                    "var_x": 0.000136701,
                                    "var_g": 12.255,
                                    "var_dx": 0.00468653,
                                    "var_dg": 1.53195}}

parameter_set = parameter_sets[data_set_name]
parameter_set

In [ ]:
# --------- SIMULATE --------- #
cells_simulated = simulate_cells(dt, n_cells, parameter_set, div_mode,
                                division_log_length, 
                                division_time, division_addition, 
                                log_length0=log_length0, gfp0=gfp0, 
                                tree=True,
                                discard_cells=discard_cells)

In [ ]:
base_dir ='../../data_sim_manuscript/{:s}_simulation{:d}_{:s}_{:s}_bleaching'

if parameter_set["beta"]>0:
    bleachin_flag="with"
else:
    bleachin_flag="wo"
    
out_dir = mk_missing_dir(base_dir.format(data_set_name, n_cells, div_mode, bleachin_flag))
out_dir

In [ ]:
# ----------- SAVE ----------- #
dt_measument = 3 # in minutes

dataset = build_data_set_scale_gfp_noise(cells_simulated, 
                                                parameter_set['var_x'], 
                                                parameter_set['var_g'], 
                                                dt_measument,
                                                atol=dt/10)

In [ ]:

dataset["length_noise"] = np.exp(dataset["log_length_noise"])
directory, filename = get_next_file_name(out_dir)

write_csv_config(os.path.join(directory, "csv_config.txt"))

write_param_file(os.path.join(directory, "parameters_sim.txt"), parameter_set) 
param_file = write_param_file_noise(os.path.join(directory, "parameters_noise.txt"), 
                                                parameter_set, 
                                                rel_step=0.1, 
                                                log_noise=0.1, fixed=["beta"]) 
dataset.to_csv(filename)



In [ ]:

dataset_full = build_data_set_scale_gfp_noise(cells_simulated[:50], 
                                                parameter_set['var_x'], 
                                                parameter_set['var_g'], 
                                                dt,
                                                atol=dt/10)



In [ ]:
filename_full = filename[:-4] + "_full.csv"
filename_full

In [ ]:

dataset_full["length_noise"] = np.exp(dataset_full["log_length_noise"])

dataset_full.to_csv(filename_full)

In [ ]:

# ----------- RUN ----------- #
run_cmd = "RealTrace" + " -c " + os.path.join(directory, "csv_config.txt") + \
    " -b " + param_file + \
    " -t " + str(1e-40) +  " -i " + filename + " -l 0 " + '-m -p' 
print("run: -> \n", run_cmd)
s = subprocess.run(run_cmd, capture_output=True, shell=True)
s

## plots


In [ ]:

# =============== PLOT =============== #
def plot_cells(cells, n_steps=1):   
    _, axes = plt.subplots(2, figsize=(5,4))
    ax = axes.ravel()

    for j in range(len(cells[::n_steps])):
        cell = copy.deepcopy(cells[j])
        cell.time = np.array(cell.time)

        if n_steps>1:
            ax[0].set_title("log length (showing every {:d}th cell)".format(n_steps))
            ax[1].set_title("gfp (showing every {:d}th cell)".format(n_steps))

        else:
            ax[0].set_title("log length")
            ax[1].set_title("gfp")

        # ax[0].set_ylim([1.2, 2.2])
        
        if len(cells[::n_steps]) <20:
            ax[0].axvline(cell.time[-1], ls='--', color='tab:blue')
            ax[1].axvline(cell.time[-1], ls='--', color='tab:orange')

        if j ==0:
            ax[0].plot(cell.time, np.exp(np.array(cell.log_length)), label='log length', color='tab:blue')
            ax[1].plot(cell.time, np.array(cell.gfp), color='tab:orange', label='gfp')

        else:
            ax[0].plot(cell.time, np.exp(np.array(cell.log_length)), color='tab:blue')
            ax[1].plot(cell.time, np.array(cell.gfp), color='tab:orange')
        

    for j in range(2):
        ax[j].legend()
    plt.tight_layout()
    plt.show()


In [ ]:
plot_cells(cells_simulated, n_steps=1)

In [ ]:
cell_sparse = df2cells(dataset)

In [ ]:

# =============== PLOT =============== #
def plot_cells_scatter(cells, n_steps=1):
    _, axes = plt.subplots(2, figsize=(5,4))
    ax = axes.ravel()

    for j in range(len(cells[::n_steps])):
        cell = copy.deepcopy(cells[j])
        cell.time = np.array(cell.time)

        if n_steps>1:
            ax[0].set_title("log length (showing every {:d}th cell)".format(n_steps))
            ax[1].set_title("gfp (showing every {:d}th cell)".format(n_steps))

        else:
            ax[0].set_title("log length")
            ax[1].set_title("gfp")

        # ax[0].set_ylim([1.2, 2.2])
        s=2
        if j ==0:
            ax[0].scatter(cell.time, np.array(cell.log_length), label='log length', color='tab:blue',s=s)
            ax[1].scatter(cell.time, np.array(cell.gfp), color='tab:orange', label='gfp',s=s)

        else:
            ax[0].scatter(cell.time, np.array(cell.log_length), color='tab:blue', s=s)
            ax[1].scatter(cell.time, np.array(cell.gfp), color='tab:orange', s=s)
    for j in range(2):
        ax[j].legend()
    plt.tight_layout()

    plt.show()

In [ ]:
plot_cells_scatter(cell_sparse[:5], n_steps=1)


In [15]:

# def add_noise_in_log(x, scale):
#     return np.exp( np.log(x) + np.random.normal(0, scale=scale) )

# def write_param_file_var_noise(filename, parameters, rel_step=0.1, fixed=[]):
#     with open(filename, "w") as fin:
#         fin.write("# Generated config file for simulated data with noise\n")
        
#         parameters_init = {}
#         parameters_init["mean_lambda"] =  add_noise_in_log(np.sqrt(parameters["var_lambda"]), 0.3)
#         parameters_init["gamma_lambda"] = add_noise_in_log(parameters["gamma_lambda"], 1)
#         parameters_init["var_lambda"] =   add_noise_in_log(np.sqrt(parameters["var_lambda"]), 1)**2
        
#         parameters_init["mean_q"] =  add_noise_in_log(np.sqrt(parameters["var_q"]), 0.3)
#         parameters_init["gamma_q"] = add_noise_in_log(parameters["gamma_q"], 1)
#         parameters_init["var_q"] =   add_noise_in_log(np.sqrt(parameters["var_q"]), 1)**2
        
#         parameters_init["var_x"] =   add_noise_in_log(np.sqrt(parameters["var_x"]), 1)**2
#         parameters_init["var_g"] =   add_noise_in_log(np.sqrt(parameters["var_g"]), 1)**2
        
#         parameters_init["var_dx"] =   add_noise_in_log(np.sqrt(parameters["var_dx"]), 1)**2
#         parameters_init["var_dg"] =   add_noise_in_log(np.sqrt(parameters["var_dg"]), 1)**2
        
#         for param_name in parameters.keys():
#             if param_name not in fixed:

#                 init = parameters_init[param_name]
#                 fin.write("{:s} = {:.2E}, {:.2E}\n".format(param_name, init, init*rel_step))
#             else:
#                 init = parameters[param_name]
#                 fin.write("{:s} = {:.2E}\n".format(param_name, init))
#         plt.scatter(parameters_init["gamma_q"], parameters_init["gamma_q"])
#     return filename


# parameters = {  "mean_lambda":1 ,        # 0.01, 
#                 "gamma_lambda": 1,       # 0.01,
#                 "var_lambda": 1,         # 1e-7,
#                 "mean_q": 1,               # 10,
#                 "gamma_q": 1,            # 0.01,
#                 "var_q": 1,               # 0.1,
#                 "beta": 1,               # 0.001, 
#                 "var_x": 1,              # 1e-3
#                 "var_g": 1,              # 5000
#                 "var_dx": 1,
#                 "var_dg": 1}

# for i in range(100):
#     write_param_file_var_noise("test.txt", parameters, 0.1, fixed="beta")